<a href="https://colab.research.google.com/github/aurora1112-j/aurora1112-j.github.io/blob/main/cold_war/py/atop5_1ddyr.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import pandas as pd
import numpy as np

# ==========================================
# 1. 读取数据
# ==========================================
file_path = 'atop5_1ddyr.dta'
# 只读取需要的列，节省内存
cols_to_keep = ['stateA', 'stateB', 'year', 'defense', 'nonagg', 'consul']
df = pd.read_stata(file_path, columns=cols_to_keep)

# ==========================================
# 2. 筛选年份 (1945 - 1991)
# ==========================================
cold_war_df = df[(df['year'] >= 1945) & (df['year'] <= 1991)].copy()

# ==========================================
# 3. 定义权重计算函数 (层级覆盖逻辑)
# ==========================================
def assign_security_weight(row):
    # 逻辑：防御 > 互不侵犯 > 协商
    if row['defense'] == 1:
        return 1.0  # 强盟友 (High Influence)
    elif row['nonagg'] == 1:
        return 0.5  # 互不侵犯 (Medium)
    elif row['consul'] == 1:
        return 0.2  # 协商 (Low)
    else:
        return 0.0  # 无连接

# 应用函数生成 'weight' 列
print("正在计算网络权重...")
cold_war_df['weight'] = cold_war_df.apply(assign_security_weight, axis=1)

# ==========================================
# 4. 清洗与输出 (Cleaning & Output)
# ==========================================
# 剔除权重为 0 的行（即没有任何关系的节点对），创建有关系的节点对的副本
edges = cold_war_df[cold_war_df['weight'] > 0].copy()

# 检查不对称性 (Asymmetry Check)
# 验证数据中是否存在 A->B 和 B->A 权重不一致的情况，即“大国影响力”假设
edges_check = edges[['stateA', 'stateB', 'year', 'weight']]
merged_check = edges_check.merge(
    edges_check,
    left_on=['stateA', 'stateB', 'year'],
    right_on=['stateB', 'stateA', 'year'],
    suffixes=('_AB', '_BA'),
    how='inner'
)
# 找出权重不相等的连边
asymmetric_dyads = merged_check[merged_check['weight_AB'] != merged_check['weight_BA']]

print("-" * 30)
print("处理完成！数据概览：")
print("-" * 30)
print(f"有效连边总数: {len(edges)}")
print(f"发现不对称连边数: {len(asymmetric_dyads)}")
print("(不对称连边代表了单向的政治影响力，例如保护国关系)")

print("\n数据前 5 行预览:")
print(edges.head())

# ==========================================
# 5. 保存结果 (Save)
# ==========================================
# 保存为 CSV 供后续建模使用
output_file = 'atop_clean.csv'
edges.to_csv(output_file, index=False)
print(f"\n已保存清洗后的数据至: {output_file}")

正在计算网络权重...
------------------------------
处理完成！数据概览：
------------------------------
有效连边总数: 76506
发现不对称连边数: 304
(不对称连边代表了单向的政治影响力，例如保护国关系)

数据前 5 行预览:
   stateA  stateB    year  defense  nonagg  consul  weight
3     2.0    20.0  1945.0      1.0     0.0     0.0     1.0
4     2.0    20.0  1949.0      1.0     1.0     1.0     1.0
5     2.0    20.0  1950.0      1.0     1.0     1.0     1.0
6     2.0    20.0  1951.0      1.0     1.0     1.0     1.0
7     2.0    20.0  1952.0      1.0     1.0     1.0     1.0

已保存清洗后的数据至: atop_clean.csv
